# Un pixel alla volta

Il codice del capitolo [«Un pixel alla volta»](https://book.paithon.it/main/VerosimiglianzaEsatta/pixel-per-pixel.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy scikit-learn torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Un pixel alla volta

[Leggi la pagina](https://book.paithon.it/main/VerosimiglianzaEsatta/pixel-per-pixel.html)


### Il codice, e una sorpresa


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


class ConvMascherata(nn.Conv2d):
    """Convoluzione che puo' guardare solo i pixel gia' visitati.

    Ordine di scansione: riga per riga, da sinistra a destra. Il tipo 'A'
    esclude anche il pixel centrale (serve al primo strato: se lo vedesse, il
    modello imparerebbe a copiarlo); il tipo 'B' lo include, perche' da li' in
    poi il centro non e' piu' il pixel vero ma un riassunto legittimo.
    """

    def __init__(self, tipo, *a, **kw):
        super().__init__(*a, **kw)
        _, _, kh, kw_ = self.weight.shape
        m = torch.ones(kh, kw_)
        m[kh // 2, kw_ // 2 + (1 if tipo == "B" else 0):] = 0   # resto della riga
        m[kh // 2 + 1:] = 0                                      # tutte le righe sotto
        self.register_buffer("maschera", m)

    def forward(self, x):
        return self._conv_forward(x, self.weight * self.maschera, self.bias)


def campo_visivo(n_strati, rif=(8, 4), n=9):
    """Quali pixel entrano DAVVERO nel conto per il pixel `rif`?

    Non lo deduciamo dalle maschere: lo chiediamo al gradiente. Se muovendo un
    pixel l'uscita in `rif` non cambia, quel pixel non e' stato guardato.
    """
    strati = [ConvMascherata("A", 1, 8, 3, padding=1), nn.ReLU()]
    for _ in range(n_strati - 2):
        strati += [ConvMascherata("B", 8, 8, 3, padding=1), nn.ReLU()]
    strati.append(ConvMascherata("B", 8, 1, 3, padding=1))
    x = torch.zeros(1, 1, n, n, requires_grad=True)
    nn.Sequential(*strati)(x)[0, 0, rif[0], rif[1]].backward()
    return x.grad[0, 0].abs() > 0


N, RIF = 9, (8, 4)
prima = torch.tensor([[(r * N + c) < (RIF[0] * N + RIF[1]) for c in range(N)]
                      for r in range(N)])

for L in (6, 12, 24):
    visto = campo_visivo(L, RIF, N)
    print(f"{L:2d} strati | pixel del futuro guardati: {int((visto & ~prima).sum())}"
          f" | pixel del passato mai guardati: {int((prima & ~visto).sum())}")

print("\ncampo visivo con 24 strati ('#' visto, '.' passato mai visto, "
      "' ' futuro):")
visto = campo_visivo(24, RIF, N)
for r in range(N):
    print("   " + " ".join("#" if visto[r, c] else ("." if prima[r, c] else " ")
                           for c in range(N)))

## Il flusso che si può invertire

[Leggi la pagina](https://book.paithon.it/main/VerosimiglianzaEsatta/flussi.html)


### Il fattore che nessuno si aspetta


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# x e' uniforme fra 0 e 1: la sua densita' vale 1 dappertutto li' dentro, e
# infatti l'area sotto la curva fa 1. Adesso stiriamo: y = 3x + 1, lo stesso
# intervallo disteso su una lunghezza tripla. La quantita' d'acqua non cambia,
# il tavolo si allarga.
x = rng.random(2_000_000)
y = 3 * x + 1

bordi = np.linspace(1, 4, 61)                      # 60 caselle su [1, 4]
misurata, _ = np.histogram(y, bins=bordi, density=True)

print(f"densita' di x (uniforme su [0,1]), misurata: {1.0:.3f}")
print(f"densita' di y (uniforme su [1,4]), misurata: {misurata.mean():.3f}")
print(f"rapporto fra le due: {1 / misurata.mean():.2f}  <- e' lo stiramento, 3")
print()
print(f"area sotto la densita' di y, col fattore:   "
      f"{(misurata * np.diff(bordi)).sum():.3f}")
print(f"area sotto la densita' di y, senza fattore: "
      f"{(1.0 * np.diff(bordi)).sum():.3f}  <- non e' una probabilita'")

### Un flusso vero, in venti righe


In [ ]:
import math

import torch
import torch.nn as nn
from sklearn.datasets import make_moons

torch.manual_seed(0)


class Accoppiamento(nn.Module):
    """Una coordinata passa intatta; l'altra viene scalata e traslata in
    funzione della prima.

    Il jacobiano e' triangolare per costruzione, quindi il suo determinante e'
    il prodotto degli elementi sulla diagonale: qui una sola scala, che il
    passo restituisce insieme al risultato. La rete che decide scala e
    traslazione NON deve essere invertibile, e infatti non lo e'.
    """

    def __init__(self, scambia, nascosto=64):
        super().__init__()
        self.scambia = scambia
        self.rete = nn.Sequential(nn.Linear(1, nascosto), nn.Tanh(),
                                  nn.Linear(nascosto, nascosto), nn.Tanh(),
                                  nn.Linear(nascosto, 2))

    def _st(self, fissa):
        s, t = self.rete(fissa.unsqueeze(1)).chunk(2, dim=1)
        return torch.tanh(s).squeeze(1), t.squeeze(1)   # tanh: scale sane

    def _ricomponi(self, fissa, mobile):
        return (torch.stack([mobile, fissa], 1) if self.scambia
                else torch.stack([fissa, mobile], 1))

    def avanti(self, x):                      # dati -> latente
        fissa, mobile = (x[:, 1], x[:, 0]) if self.scambia else (x[:, 0], x[:, 1])
        s, t = self._st(fissa)
        return self._ricomponi(fissa, mobile * torch.exp(s) + t), s

    def indietro(self, z):                    # latente -> dati
        fissa, mobile = (z[:, 1], z[:, 0]) if self.scambia else (z[:, 0], z[:, 1])
        s, t = self._st(fissa)
        return self._ricomponi(fissa, (mobile - t) * torch.exp(-s))


class Flusso(nn.Module):
    """Sei accoppiamenti a turni alterni: cosi' ogni coordinata viene
    trasformata e ogni coordinata fa da guida."""

    def __init__(self, n=6):
        super().__init__()
        self.passi = nn.ModuleList(Accoppiamento(i % 2 == 1) for i in range(n))

    def avanti(self, x):
        logdet = torch.zeros(len(x))
        for p in self.passi:
            x, s = p.avanti(x)
            logdet = logdet + s
        return x, logdet

    def indietro(self, z):
        for p in reversed(self.passi):
            z = p.indietro(z)
        return z

    def log_densita(self, x):
        """Il cambio di variabile, scritto: log p(x) = log p(z) + log|det|."""
        z, logdet = self.avanti(x)
        log_gauss = -0.5 * (z ** 2).sum(1) - math.log(2 * math.pi)
        return log_gauss + logdet


X, _ = make_moons(2000, noise=0.06, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0)

flusso = Flusso()
opt = torch.optim.Adam(flusso.parameters(), lr=3e-3)
for passo in range(1500):
    perdita = -flusso.log_densita(X).mean()          # verosimiglianza, e basta
    opt.zero_grad(); perdita.backward(); opt.step()
print(f"log-verosimiglianza media per punto: {-perdita.item():.3f} nat")

# --- Prova 1: e' davvero invertibile? Andata e ritorno, e si controlla.
with torch.no_grad():
    z, _ = flusso.avanti(X)
    errore = (flusso.indietro(z) - X).abs().max().item()
print(f"errore massimo andata e ritorno: {errore:.2e}")

# --- Prova 2: e' davvero una densita'? Si integra su una griglia fitta.
# In due dimensioni la quadratura si puo' ancora fare, e vale come verifica
# del fatto che il fattore |det| non e' decorativo: senza, non farebbe 1.
g = torch.linspace(-6, 6, 601)
gx, gy = torch.meshgrid(g, g, indexing="ij")
griglia = torch.stack([gx.reshape(-1), gy.reshape(-1)], 1)
with torch.no_grad():
    p = flusso.log_densita(griglia).exp()
area = (g[1] - g[0]) ** 2
print(f"integrale della densita' sulla griglia: {(p.sum() * area).item():.4f}")

# --- Prova 3: la densita' distingue le lune dal resto del piano?
fuori = torch.rand(2000, 2) * 8 - 4
with torch.no_grad():
    print(f"log-densita' media sulle lune:  {flusso.log_densita(X).mean():.2f}")
    print(f"log-densita' media a caso:      {flusso.log_densita(fuori).mean():.2f}")